# Parquet WellViz DataFrame Inspection

This notebook inspects the indexed SCREEN WellViz Parquet package with tabular previews, profiling summaries, and visual quality-control plots.

## 1. Import Libraries and Configure Display

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import plotly.express as px
    import plotly.graph_objects as go
    PLOTLY_AVAILABLE = True
except ImportError:
    PLOTLY_AVAILABLE = False

pd.set_option("display.max_rows", 30)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
pd.set_option("display.precision", 4)
sns.set_theme(style="whitegrid")

print("Plotly available:", PLOTLY_AVAILABLE)

## 2. Load or Create Example DataFrames

The default path points to the indexed baseline package generated by SCREEN. If it is unavailable, the notebook creates a small synthetic fallback so the inspection sections still run.

In [ ]:
package_dir = Path("work/results/baseline_wellviz_indexed")
parquet_path = package_dir / "data.parquet"

if parquet_path.exists():
    df = pd.read_parquet(parquet_path)
    source_frames = {
        source: frame.reset_index(drop=True)
        for source, frame in df.groupby("source", sort=True)
    }
    print(f"Loaded {len(df):,} rows from {parquet_path}")
else:
    rng = np.random.default_rng(42)
    df = pd.DataFrame({
        "source": rng.choice(["INIT", "UNRST"], 500),
        "property": rng.choice(["PORV", "PRESSURE", "SWAT"], 500),
        "j_column": rng.integers(0, 3, 500),
        "x": rng.integers(0, 10, 500),
        "z": rng.integers(0, 25, 500),
        "value": rng.normal(size=500),
        "i": rng.integers(0, 10, 500),
        "j": rng.integers(0, 3, 500),
        "k": rng.integers(0, 25, 500),
    })
    source_frames = {source: frame.copy() for source, frame in df.groupby("source", sort=True)}
    print("Parquet package not found; using synthetic fallback data.")

df.head()

## 3. Quick Structural Inspection